In [ ]:
import pandas as pd        

# confirm the working directory and file existence
import os
print("cwd:", os.getcwd())
print("exists:", os.path.isfile("personal_finance_dataset.xlsx"))

# read the 'datathon_finance' sheet from the Excel file
df = pd.read_excel("personal_finance_dataset.xlsx", sheet_name="datathon_finance")

column_mapping = {
    'PAGEMIEG': 'Age_Group',
    'PATTCRU':  'Credit_Card_Payment_Behavior',
    'PATTSITC': 'COVID_Financial_Impact',
    'PATTSKP':  'Skipped_Payments',
    'PEDUCMIE': 'Education_Level',
    'PEFATINC': 'Annual_After_Tax_Income',
    'PFMTYPG':  'Family_Type',
    'PFTENUR':  'Home_Ownership',
    'PLFFPTME': 'Work_Status_2022',
    'PNBEARG':  'Number_of_Earners',
    'PPVRES':   'Province',
    'PWAPRVAL': 'Home_Value',
    'PWASTDEP': 'Bank_Deposits',
    'PWATFS':   'TFSA_Balance',
    'PWDPRMOR': 'Mortgage_Debt',
    'PWDSLOAN': 'Student_Loan_Debt',
    'PWDSTCRD': 'Credit_Card_Debt',
    'PWDSTLOC': 'Line_of_Credit_Debt',
    'PWNETWPG': 'Net_Worth'
}

df.rename(columns=column_mapping, inplace=True)

print(df.head())

In [ ]:
import numpy as np

def define_resilience(df, shock_percentage=0.15):
    # 1. ANNUAL EXPENSE & LIQUIDITY
    df['Total_Liquidity'] = df['Bank_Deposits'] + df['TFSA_Balance']

    # Annual Expense Proxy (60% of income)
    df['Annual_Expenses_Base'] = (df['Annual_After_Tax_Income'] * 0.6)
    
    # Calculate Runway in years (How many years they can cover expenses with liquid assets if income drops to zero)
    df['Years_of_Runway'] = df['Total_Liquidity'] / df['Annual_Expenses_Base']
    
    # 2. HOUSING BURDEN
    # Owners (Home_Ownership == 2): Proxy 4% of Mortgage Debt as annual cost
    # Renters (Home_Ownership == 3): Proxy 30% of income as annual rent
    df['Est_Annual_Housing_Cost'] = np.where(
        df['Home_Ownership'] == 2, 
        df['Mortgage_Debt'] * 0.04, 
        np.where(df['Home_Ownership'] == 3, df['Annual_After_Tax_Income'] * 0.30, 0))
    
    # Housing Burden Ratio (Costs / Income)
    df['Housing_Burden_Ratio'] = df['Est_Annual_Housing_Cost'] / df['Annual_After_Tax_Income'].replace(0, 1)

    # 3. CONSUMER DEBT BURDEN
    # Calculate Total Non-Mortgage Debt (The "Consumer Debt")
    df['Total_Consumer_Debt'] = df['Credit_Card_Debt'] + df['Line_of_Credit_Debt'] + df['Student_Loan_Debt']

    # Annual Debt-to-Income Ratio
    # A ratio > 0.5 is often considered "serious financial strain" in Canadian banking
    df['DTI_Ratio'] = df['Total_Consumer_Debt'] / df['Annual_After_Tax_Income'].replace(0, 1)

    # 4. THE "AT-RISK" TARGET VARIABLE
    # "At Risk" Baseline (No shock): Less than 3 months (0.25 yrs) runway OR High Housing Burden (>35%) OR High DTI Ratio (>50%) OR Skipped Payments
    df['Is_At_Risk_Base'] = np.where(
        (df['Years_of_Runway'] < 0.25) | 
        (df['Housing_Burden_Ratio'] > 0.35) | 
        (df['DTI_Ratio'] > 0.50) |             
        (df['Skipped_Payments'] == 1), 1, 0)
    
    # --- 4. ECONOMIC SHOCK SIMULATION ---
    # Increase expenses by the shock percentage (e.g., 1.15 for a 15% increase in inflation)
    df['Annual_Expenses_Shocked'] = df['Annual_Expenses_Base'] * (1 + shock_percentage)
    
    # Re-calculate runway with the inflated expenses
    df['Runway_Post_Shock'] = df['Total_Liquidity'] / (df['Annual_Expenses_Shocked'])
    
    # Define who is at risk under the shock
    df['Is_At_Risk_Shocked'] = np.where(df['Runway_Post_Shock'] < 0.25, 1, 0)
    
    # Calculate the number of people who were safe but are now at risk after the shock
    df['Became_Vulnerable'] = ((df['Is_At_Risk_Base'] == 0) & (df['Is_At_Risk_Shocked'] == 1)).astype(int)
    
    return df

df = define_resilience(df, shock_percentage=0.15)


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

# Summary Table by Age Group
age_labels = {1: '18-24', 2: '25-34', 3: '35-44', 4: '45-54'}
summary_table = df.groupby('Age_Group')[['Is_At_Risk_Base', 'Became_Vulnerable']].mean() * 100
summary_table.index = summary_table.index.map(age_labels)

print("Financial Fragility by Age Group (%):")
print(summary_table)


In [ ]:
# Visualize the % of each age group that became vulnerable after the shock
plt.figure(figsize=(10, 6))
sns.barplot(x=summary_table.index, y=summary_table['Became_Vulnerable'], palette='Reds_r')
plt.title('The "Fragile" Class: % Pushed into Risk by 15% Inflation Shock', fontsize=14)
plt.ylabel('% of Group Newly Vulnerable')
plt.show()

In [ ]:
# What correlates most with being 'At Risk'?
correlations = df[['Is_At_Risk_Base', 'Annual_After_Tax_Income', 'Total_Liquidity', 
                  'Housing_Burden_Ratio', 'Student_Loan_Debt', 'Credit_Card_Debt']].corr()['Is_At_Risk_Base'].sort_values()

print("Top Risk Drivers (Correlation):")
print(correlations)

In [ ]:
# Feature Importance Analysis
# 1. Select the features (X) and the target (y)
# We want to see what predicts the "Base Risk"
features = ['Age_Group', 'Annual_After_Tax_Income', 'Total_Liquidity', 
            'Housing_Burden_Ratio', 'Student_Loan_Debt', 'Credit_Card_Debt', 'DTI_Ratio']

x = df[features]
y = df['Is_At_Risk_Base']

# 2. Train the Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(x, y)

# 2. Train the Model
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(x, y)

# 3. Create the importance_df
importance_df = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_
}).sort_values(by='importance', ascending=False)

# 4. NOW call your plot function (Fixed the .sh to .show())
def plot_importance(importance_df):
    plt.figure(figsize=(10, 6))
    sns.barplot(x='importance', y='feature', data=importance_df, palette='viridis')
    plt.title("What Drives Financial Risk?", fontsize=16, fontweight='bold')
    plt.xlabel("Predictive Power (Feature Importance)", fontsize=12)
    plt.ylabel("")
    plt.tight_layout()
    plt.show()

plot_importance(importance_df)